# Step 1 — Feature extraction and matching, on a Colab GPU

This notebook does the **GPU-heavy half** of photogrammetry: it finds features in
your photos, then matches them across image pairs. The output is a single
`database.db` file.

After this, run **step 2** (the desktop app) to build camera positions, then
**step 3** to train Gaussian Splatting.

**Before you run anything:** `Runtime` → `Change runtime type` → **T4 GPU**.

### Why this notebook exists

The `colmap` package from `apt` is compiled **without CUDA**. On CPU, exhaustive
matching of 320 photos takes over 11 hours. On a T4 it takes 28 minutes — the
same work, about 23× faster per image pair.

This notebook uses a prebuilt COLMAP 3.13.0 with CUDA enabled:
https://github.com/Ryanhuhut/colmap-cuda-colab

## 1. Configuration — edit this cell

In [ ]:
# ==================== EDIT THIS CELL ====================

# Zip file on Google Drive holding your photos.
# The layout inside the zip does not matter: the photos can sit at the top
# level, or inside one folder, or inside several nested folders. Everything is
# gathered into one flat folder for you, and __MACOSX / hidden files are ignored.
#
# How to get this path: click the folder icon in Colab's left sidebar, open
# drive/MyDrive, find your zip, click the three dots next to it, "Copy path".
# Then paste it here between the quotes.
IMAGES_ZIP = "/content/drive/MyDrive/CHANGE_ME.zip"

# Short name for this scan. The database is saved as PROJECT.db, so every scan
# ends up with its own file — "database.db" for all of them means the second
# download quietly overwrites the first, or you feed the wrong one to step 2 and
# spend two hours reconstructing somebody else's photos.
# Leave it empty to take the name from the zip: "Meo_1600.zip" -> "Meo_1600.db".
PROJECT = ""

# Folder on Drive where the finished database will be saved.
# It is created for you if it does not exist yet.
OUTPUT_DIR = "/content/drive/MyDrive/img3dpl"

# "exhaustive"  compares every possible pair of photos. Best quality — it catches
#               loop closures, so the model does not drift. Needs a GPU.
# "sequential"  compares each photo with its 10 neighbours only. Roughly 17x less
#               work, but it misses loop closures when you orbit an object.
MATCHER = "exhaustive"

# Camera model. OPENCV suits phone cameras.
CAMERA_MODEL = "OPENCV"

# True when every photo came from the same camera at the same zoom.
# Keep this True for a folder shot with one lens at one focal length — it also
# rescues resized photos, whose EXIF focal length is usually stripped, because
# every photo then shares one set of guessed camera parameters.
SINGLE_CAMERA = True

# ========================================================
import os
import re
import unicodedata

if "CHANGE_ME" in IMAGES_ZIP:
    raise SystemExit("Edit IMAGES_ZIP above to point at your own zip on Drive.")


def short_name(path):
    """Project name taken from a file name: "Mèo_1600.zip" -> "Meo_1600".

    Accents and spaces are stripped because this name becomes a file name and
    goes onto a command line, where anything unusual turns into a puzzling error
    hours later.
    """
    name = os.path.basename(path.rstrip("/"))
    for suffix in (".tar.gz", ".tgz", ".tar", ".zip"):
        if name.lower().endswith(suffix):
            name = name[: -len(suffix)]
            break
    name = name.replace("đ", "d").replace("Đ", "D")
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    return name or "scan"


PROJECT = PROJECT.strip() or short_name(IMAGES_ZIP)

print("Photos  :", IMAGES_ZIP)
print("Database:", f"{OUTPUT_DIR}/{PROJECT}.db")
print("Matcher :", MATCHER)

## 2. Setup — GPU, COLMAP, photos

In [ ]:
import os
import shutil
import subprocess
import sys

COLMAP_URL = ("https://github.com/Ryanhuhut/colmap-cuda-colab/releases/latest/"
              "download/colmap-3.13.0-cuda12.2-ubuntu2204-sm75.tar.gz")
IMAGES_DIR = "/content/images"      # flat folder COLMAP reads
UNZIP_DIR = "/content/_unzipped"    # whatever shape the zip happened to have
EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

os.environ["PATH"] = "/opt/colmap-cuda/bin:" + os.environ["PATH"]

# ---- GPU -------------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                      "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU.")

gpu_name = gpu.stdout.strip()
print("GPU:", gpu_name)

# This COLMAP build targets sm_75 only, which is the T4. On any other card it
# fails with a CUDA architecture error. Better to say so now than 20 minutes in.
if "T4" not in gpu_name:
    print()
    print("!! WARNING — this COLMAP build only supports the Tesla T4 (sm_75).")
    print(f"!! You were given: {gpu_name}")
    print("!! Matching will fail. Rebuild COLMAP with CUDA_ARCH=\"75;80;89\";")
    print("!! see build-colmap/ in the repository.")

# ---- COLMAP ----------------------------------------------------------------
if not os.path.exists("/opt/colmap-cuda/bin/colmap"):
    print("\nInstalling COLMAP (about 7 seconds)...")
    subprocess.run(["wget", "-q", COLMAP_URL, "-O", "/tmp/colmap.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/colmap.tar.gz", "-C", "/opt"], check=True)

version = subprocess.run(["colmap", "-h"], capture_output=True, text=True).stdout
print("\n".join(version.splitlines()[:2]))
if "with CUDA" not in version:
    raise SystemExit("This COLMAP has no CUDA support — wrong package?")

# ---- Photos ----------------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")


def find_images(root):
    """Every image under root, at any depth.

    Zip tools add folders nobody wants: __MACOSX holds resource forks whose
    names start with "._", and those are not real photos. Skipping them here
    keeps COLMAP from choking on files it cannot decode.
    """
    found = []
    for folder, subdirs, files in os.walk(root):
        subdirs[:] = [d for d in subdirs
                      if d != "__MACOSX" and not d.startswith(".")]
        for name in sorted(files):
            if name.startswith(".") or not name.lower().endswith(EXTENSIONS):
                continue
            found.append(os.path.join(folder, name))
    return sorted(found)


if not os.path.isdir(IMAGES_DIR) or not os.listdir(IMAGES_DIR):
    if not os.path.isfile(IMAGES_ZIP):
        raise SystemExit(f"Zip not found on Drive: {IMAGES_ZIP}")

    print("\nExtracting photos...")
    shutil.rmtree(UNZIP_DIR, ignore_errors=True)
    os.makedirs(UNZIP_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", IMAGES_ZIP, "-d", UNZIP_DIR], check=True)

    # The zip may be flat, or — far more common, since you zip a folder rather
    # than a selection of files — it may hold one folder of photos, or several
    # nested ones. Flatten whatever came out into a single folder, because both
    # this notebook and steps 2-3 expect one flat image directory.
    found = find_images(UNZIP_DIR)
    if not found:
        print("\nNo images inside the zip. Here is what it did contain:")
        for folder, _, files in os.walk(UNZIP_DIR):
            rel = os.path.relpath(folder, UNZIP_DIR)
            print(f"  {rel}/  ({len(files)} files)")
            for name in sorted(files)[:5]:
                print(f"      {name}")
        raise SystemExit("Zip holds no .jpg/.png/.tif files — wrong zip?")

    os.makedirs(IMAGES_DIR, exist_ok=True)
    taken = set()
    for src in found:
        name = os.path.basename(src)
        if name in taken:
            # Two sub-folders can each hold a DSC_0001.jpg. Keep both, since
            # dropping one silently would quietly lose a viewpoint.
            stem, ext = os.path.splitext(name)
            parent = os.path.basename(os.path.dirname(src))
            name = f"{parent}_{stem}{ext}"
            n = 2
            while name in taken:
                name = f"{parent}_{stem}_{n}{ext}"
                n += 1
        taken.add(name)
        shutil.move(src, os.path.join(IMAGES_DIR, name))

    depth = max(p[len(UNZIP_DIR):].count(os.sep) for p in found)
    if depth > 1:
        print(f"Photos were nested inside sub-folders — flattened into {IMAGES_DIR}")
    shutil.rmtree(UNZIP_DIR, ignore_errors=True)

PHOTOS = sorted(f for f in os.listdir(IMAGES_DIR)
                if f.lower().endswith(EXTENSIONS))
N_PHOTOS = len(PHOTOS)
print(f"\nPhotos ready: {N_PHOTOS}")
if N_PHOTOS == 0:
    raise SystemExit(f"{IMAGES_DIR} is empty. Re-run after fixing IMAGES_ZIP.")
if N_PHOTOS < 20:
    print("Only a handful of photos — reconstruction usually needs 20-30 at least.")

# Sizes, from a sample. All photos should share one resolution: they came from
# one lens at one focal length, and a single resize pass. A mixture means the
# folder mixes sources, and SINGLE_CAMERA above would then be wrong.
from PIL import Image

sizes = set()
for name in PHOTOS[:: max(1, N_PHOTOS // 30)]:
    with Image.open(os.path.join(IMAGES_DIR, name)) as im:
        sizes.add(im.size)

print("Resolution  :", ", ".join(f"{w}x{h}" for w, h in sorted(sizes)))
if len(sizes) > 2:  # portrait + landscape from one camera is fine, more is not
    print("!! Several different resolutions — if these came from different")
    print("!! cameras or zoom levels, set SINGLE_CAMERA = False and re-run cell 1.")

longest = max(max(s) for s in sizes)
if longest > 3200:
    print(f"!! Longest edge is {longest}px. COLMAP downscales to 3200px anyway,")
    print("!! so you are paying upload time for pixels it will throw away.")

# Rough idea of the work ahead, so nobody thinks it has frozen.
pairs = N_PHOTOS * (N_PHOTOS - 1) // 2 if MATCHER == "exhaustive" else N_PHOTOS * 10
print(f"Pairs to match: {pairs:,}   (about {pairs * 0.033 / 60:.0f} minutes on a T4)")

## 3. Extract features and match

Output is streamed live, so you can watch it move. Do not close this tab.

`exhaustive` matching on 320 photos took **28 minutes** on a free-tier T4.

In [ ]:
import os
import shutil
import subprocess
import time

WORK = "/content/colmap_work"
DB = f"{WORK}/database.db"

# Start from an empty folder every time. Reusing a database built by a different
# COLMAP version, or by a different set of photos, produces confusing
# "SQL logic error" failures much later on.
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)


def run(label, cmd):
    """Run a COLMAP command and stream its output live."""
    print(f"\n{'=' * 64}\n{label}\n{'=' * 64}", flush=True)
    started = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    minutes = (time.time() - started) / 60
    print(f"\n>>> {label} — {minutes:.1f} min, exit code {code}", flush=True)
    if code != 0:
        raise RuntimeError(f"{label} failed. Read the output above.")
    return minutes


t_extract = run("1 of 2 — Feature extraction", [
    "colmap", "feature_extractor",
    "--database_path", DB,
    "--image_path", IMAGES_DIR,
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
    "--FeatureExtraction.use_gpu", "1",
])

t_match = run(f"2 of 2 — Matching ({MATCHER})", [
    "colmap", f"{MATCHER}_matcher",
    "--database_path", DB,
    "--FeatureMatching.use_gpu", "1",
])

print(f"\nTotal: {t_extract + t_match:.1f} minutes")

## 4. Save the database to Drive

This runs automatically. The database lives on the Colab machine, which is wiped
the moment the session ends — so it is copied to Drive straight away.

It is saved as `PROJECT.db`, named after your zip, rather than `database.db`.
Two scans, two files, no way to mix them up in your downloads folder.

In [ ]:
import os
import shutil
import sqlite3

os.makedirs(OUTPUT_DIR, exist_ok=True)
saved = os.path.join(OUTPUT_DIR, f"{PROJECT}.db")

# Named after the scan, not "database.db". Once you have run two scans, a folder
# of identical file names is how the wrong database ends up in step 2 — and you
# only find out two hours later, when the model comes out as noise.
if os.path.exists(saved):
    print(f"!! {PROJECT}.db already exists on Drive and is being replaced.")
    print("!! Set PROJECT in cell 1 if that was a different scan.\n")

shutil.copy(DB, saved)

with sqlite3.connect(f"file:{saved}?mode=ro", uri=True) as conn:
    images = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
    pairs = conn.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]

print(f"Saved: {saved}")
print(f"  size   : {os.path.getsize(saved) / 1e9:.1f} GB")
print(f"  images : {images}")
print(f"  pairs  : {pairs:,}")
print()
print(f"Next: download {PROJECT}.db, then open the desktop app to build camera")
print("positions. Feed it the same 1600px photo folder you uploaded here — the")
print("camera parameters inside this database describe those exact images.")
print("Building positions runs on the CPU — a GPU does not help there, and your")
print("own machine most likely has more CPU cores than free-tier Colab.")